<!-- project-navigation -->
# 18_Preprocessing_Deployment
Role: Deployment artifacts and schema.

See [notebook guide](README.md) and [model comparison](../reports/model_comparison.md). Historical code and outputs are retained; run with the project root as working directory.


In [1]:
# ============================================================
# 18_Preprocessing_Deployment.ipynb
# PRÉPARATION AU DÉPLOIEMENT DU MODÈLE LIGHTGBM
# ============================================================

import os
import re
import joblib
import numpy as np
import pandas as pd

print("✓ Librairies importées")

✓ Librairies importées


In [2]:
# ============================================================
# CHARGEMENT DU MODÈLE FINAL
# ============================================================

MODEL_PATH = "data/models/lightgbm_scale_pos_weight.pkl"

model_lgb_spw = joblib.load(MODEL_PATH)

print("✓ Modèle chargé")
print("Type :", type(model_lgb_spw))

✓ Modèle chargé
Type : <class 'lightgbm.sklearn.LGBMClassifier'>


In [3]:
# ============================================================
# CHARGEMENT DES DONNÉES
# ============================================================

X_train, X_valid, y_train, y_valid = joblib.load(
    "data/processed/xgb_split.pkl"
)

print("X_train :", X_train.shape)
print("X_valid :", X_valid.shape)
print("Nombre de features :", X_train.shape[1])

X_train : (246008, 240)
X_valid : (61503, 240)
Nombre de features : 240


In [4]:
# ============================================================
# SAUVEGARDE DES FEATURES DU MODÈLE
# ============================================================

feature_names = X_train.columns.tolist()

joblib.dump(
    feature_names,
    "data/models/lightgbm_feature_names.pkl"
)

print("✓ Liste des 240 features sauvegardée")
print("Nombre de features :", len(feature_names))
print("\nPremières features :")
print(feature_names[:20])

✓ Liste des 240 features sauvegardée
Nombre de features : 240

Premières features :
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL']


In [5]:
# ============================================================
# VÉRIFICATION DE LA STRUCTURE
# ============================================================

print("Nombre de features :", len(feature_names))

print(
    "Toutes les features sont uniques :",
    len(feature_names) == len(set(feature_names))
)

print(
    "Features du train = features validation :",
    list(X_train.columns) == list(X_valid.columns)
)

Nombre de features : 240
Toutes les features sont uniques : True
Features du train = features validation : True


In [6]:
# ============================================================
# TYPES DES FEATURES
# ============================================================

feature_dtypes = X_train.dtypes.astype(str).to_dict()

joblib.dump(
    feature_dtypes,
    "data/models/lightgbm_feature_dtypes.pkl"
)

print("✓ Types des features sauvegardés")

print("\nRépartition des types :")
print(X_train.dtypes.value_counts())

✓ Types des features sauvegardés

Répartition des types :
int64      136
float64    104
Name: count, dtype: int64


In [7]:
# ============================================================
# VALEURS MANQUANTES
# ============================================================

missing_train = X_train.isnull().sum()
missing_valid = X_valid.isnull().sum()

print("Valeurs manquantes dans X_train :", missing_train.sum())
print("Valeurs manquantes dans X_valid :", missing_valid.sum())

if missing_train.sum() == 0 and missing_valid.sum() == 0:
    print("✓ Aucune valeur manquante")
else:
    print("⚠ Des valeurs manquantes existent")

Valeurs manquantes dans X_train : 0
Valeurs manquantes dans X_valid : 0
✓ Aucune valeur manquante


In [8]:
# ============================================================
# VALEURS INFINIES
# ============================================================

inf_train = np.isinf(X_train.select_dtypes(include=np.number)).sum().sum()
inf_valid = np.isinf(X_valid.select_dtypes(include=np.number)).sum().sum()

print("Inf dans X_train :", inf_train)
print("Inf dans X_valid :", inf_valid)

if inf_train == 0 and inf_valid == 0:
    print("✓ Aucune valeur infinie")
else:
    print("⚠ Des valeurs infinies existent")

Inf dans X_train : 0
Inf dans X_valid : 0
✓ Aucune valeur infinie


In [9]:
# ============================================================
# MÉTADONNÉES DU MODÈLE
# ============================================================

model_metadata = {
    "model_name": "LightGBM",
    "model_variant": "scale_pos_weight",
    "n_features": X_train.shape[1],
    "n_train_rows": X_train.shape[0],
    "n_valid_rows": X_valid.shape[0],
    "final_threshold": 0.68,
    "roc_auc": 0.7833,
    "ks": 0.4298
}

joblib.dump(
    model_metadata,
    "data/models/lightgbm_metadata.pkl"
)

print("✓ Métadonnées sauvegardées")
print(model_metadata)

✓ Métadonnées sauvegardées
{'model_name': 'LightGBM', 'model_variant': 'scale_pos_weight', 'n_features': 240, 'n_train_rows': 246008, 'n_valid_rows': 61503, 'final_threshold': 0.68, 'roc_auc': 0.7833, 'ks': 0.4298}


In [10]:
# ============================================================
# TEST DU MODÈLE
# ============================================================

X_test_sample = X_valid.iloc[:5].copy()

proba_test = model_lgb_spw.predict_proba(
    X_test_sample
)[:, 1]

prediction_test = (
    proba_test >= 0.68
).astype(int)

results_test = pd.DataFrame({
    "Probability_Default": proba_test,
    "Prediction": prediction_test
})

print(results_test)

   Probability_Default  Prediction
0             0.242224           0
1             0.427374           0
2             0.788322           1
3             0.297311           0
4             0.444690           0


In [11]:
# ============================================================
# TEST DE REPRODUCTIBILITÉ
# ============================================================

saved_feature_names = joblib.load(
    "data/models/lightgbm_feature_names.pkl"
)

X_test_reordered = X_valid[saved_feature_names].iloc[:5]

proba_reproduced = model_lgb_spw.predict_proba(
    X_test_reordered
)[:, 1]

print(
    "Prédictions identiques :",
    np.allclose(
        proba_test,
        proba_reproduced
    )
)

Prédictions identiques : True


In [12]:
import os

print("============================================================")
print("FICHIERS DISPONIBLES")
print("============================================================")

for root, dirs, files in os.walk("data"):
    for file in files:
        print(os.path.join(root, file))

FICHIERS DISPONIBLES
data\models\lightgbm_feature_dtypes.pkl
data\models\lightgbm_feature_names.pkl
data\models\lightgbm_metadata.pkl
data\models\lightgbm_scale_pos_weight.pkl
data\processed\application_clean.csv
data\processed\application_reduit.csv
data\processed\application_test_clean.csv
data\processed\application_test_encode.csv
data\processed\application_test_processed.csv
data\processed\application_train_encode.csv
data\processed\application_train_loaded.csv
data\processed\bureau_balance_features.csv
data\processed\bureau_features.csv
data\processed\credit_card_features.csv
data\processed\installments_features.csv
data\processed\lightgbm_ks_curve.png
data\processed\lightgbm_roc_curve.png
data\processed\model_evaluation_thresholds.csv
data\processed\model_final_evaluation.csv
data\processed\pos_cash_features.csv
data\processed\previous_application_features.csv
data\processed\shap_clients_examples.csv
data\processed\shap_feature_importance.csv
data\processed\shap_global_importance

In [13]:
print("============================================================")
print("VARIABLES / OBJETS IMPORTANTS")
print("============================================================")

print("model :", type(model_lgb_spw) if "model_lgb_spw" in globals() else "NON CHARGÉ")
print("X_train :", X_train.shape if "X_train" in globals() else "NON CHARGÉ")
print("X_valid :", X_valid.shape if "X_valid" in globals() else "NON CHARGÉ")
print("explainer :", type(explainer) if "explainer" in globals() else "NON CHARGÉ")
print("predict_with_explain :", "OUI" if "predict_with_explain" in globals() else "NON CHARGÉ")

VARIABLES / OBJETS IMPORTANTS
model : <class 'lightgbm.sklearn.LGBMClassifier'>
X_train : (246008, 240)
X_valid : (61503, 240)
explainer : NON CHARGÉ
predict_with_explain : NON CHARGÉ


In [14]:
import pandas as pd
import joblib

X_train, X_valid, y_train, y_valid = joblib.load(
    "data/processed/xgb_split.pkl"
)

print("X_valid :", X_valid.shape)

train_final = pd.read_csv(
    "data/processed/train_final.csv"
)

print("train_final :", train_final.shape)

print("SK_ID_CURR présent ?",
      "SK_ID_CURR" in train_final.columns)

if "SK_ID_CURR" in train_final.columns:
    print("\nPremiers SK_ID_CURR :")
    print(train_final["SK_ID_CURR"].head(10).tolist())

    print("\nNombre de SK_ID_CURR uniques :")
    print(train_final["SK_ID_CURR"].nunique())

X_valid : (61503, 240)
train_final : (307511, 228)
SK_ID_CURR présent ? True

Premiers SK_ID_CURR :
[100002, 100003, 100004, 100006, 100007, 100008, 100009, 100010, 100011, 100012]

Nombre de SK_ID_CURR uniques :
307511


In [16]:
import pandas as pd

train_encoded = pd.read_csv(
    "data/processed/train_encoded.csv"
)

print("train_encoded :", train_encoded.shape)

print("\nSK_ID_CURR présent ?",
      "SK_ID_CURR" in train_encoded.columns)

print("\nNombre de features X_valid :",
      X_valid.shape[1])

print("\nDifférences :")

x_features = set(X_valid.columns)
encoded_features = set(train_encoded.columns)

print("Features X_valid absentes de train_encoded :")
print(
    list(x_features - encoded_features)
)

print("\nColonnes train_encoded absentes de X_valid :")
print(
    list(encoded_features - x_features)
)

train_encoded : (307511, 242)

SK_ID_CURR présent ? True

Nombre de features X_valid : 240

Différences :
Features X_valid absentes de train_encoded :
[]

Colonnes train_encoded absentes de X_valid :
['TARGET', 'SK_ID_CURR']


In [17]:
from sklearn.model_selection import train_test_split

# ============================================================
# RECONSTRUIRE EXACTEMENT LE SPLIT
# ============================================================

train_encoded = pd.read_csv(
    "data/processed/train_encoded.csv"
)

# 240 features utilisées par le modèle
feature_columns = X_valid.columns.tolist()

X_all = train_encoded[feature_columns]
y_all = train_encoded["TARGET"]
ids_all = train_encoded["SK_ID_CURR"]


# ============================================================
# REFAIRE LE MÊME SPLIT
# ============================================================

X_train_rebuilt, X_valid_rebuilt, y_train_rebuilt, y_valid_rebuilt = train_test_split(
    X_all,
    y_all,
    test_size=0.2,
    stratify=y_all,
    random_state=42
)


# ============================================================
# RÉCUPÉRER LES IDs DES CLIENTS VALIDATION
# ============================================================

# IMPORTANT :
# train_test_split conserve les indices d'origine

valid_indices = X_valid_rebuilt.index

valid_client_ids = ids_all.loc[valid_indices].reset_index(drop=True)


# ============================================================
# VÉRIFICATIONS
# ============================================================

print("X_valid original :", X_valid.shape)
print("X_valid reconstruit :", X_valid_rebuilt.shape)

print()

print("Nombre de clients validation :", len(valid_client_ids))

print()

print("Premiers vrais SK_ID_CURR :")
print(valid_client_ids.head(10).tolist())

print()

print("Derniers vrais SK_ID_CURR :")
print(valid_client_ids.tail(10).tolist())

print()

print("Nombre d'IDs uniques :")
print(valid_client_ids.nunique())

X_valid original : (61503, 240)
X_valid reconstruit : (61503, 240)

Nombre de clients validation : 61503

Premiers vrais SK_ID_CURR :
[396899, 322041, 220127, 251531, 345558, 251419, 433604, 268026, 218614, 394494]

Derniers vrais SK_ID_CURR :
[259943, 317055, 341691, 372036, 198767, 118908, 441326, 371501, 272785, 411571]

Nombre d'IDs uniques :
61503
